# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [25]:
# imports
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display
import os

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA ='llama3.2'

In [ ]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key:
    print("API key looks good so far")
else:
    print("No API key found")

# Use a dedicated name so a later Ollama cell does not overwrite this client.
client_openai = OpenAI()

API key looks good so far


In [33]:
# Get gpt-4o-mini to answer, with streaming
def stream_answer(ques):
    stream = client_openai.chat.completions.create(
        model=MODEL_GPT,
        messages= [
            {"role": "system", "content": "You are an assisstant that will give explanation of the question asked"},
            {"role": "user", "content": ques}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id = display_handle.display_id)



In [34]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [38]:
stream_answer(question)

This line of code uses the `yield from` expression, which is a feature in Python that allows a generator to yield values from another iterator or generator. Let's break down the code step by step:

1. **Set Comprehension**: 
   The expression `{book.get("author") for book in books if book.get("author")}` is a set comprehension that creates a set of authors. Here's how it works:
   - It iterates over a collection called `books`, which is expected to be a list (or any iterable) of dictionaries, where each dictionary represents a book.
   - For each `book`, it calls `book.get("author")`, which retrieves the value associated with the key `"author"`. If the key doesn't exist, `get` returns `None`.
   - The `if book.get("author")` condition ensures that only books with a valid (non-None) author are included in the set.
   - As a result, this set comprehension extracts the authors from the books and creates a set of unique authors (since sets cannot contain duplicate values).

2. **`yield from`**: 
   The `yield from` statement is used in the context of generator functions. It allows you to yield all values from an iterable (in this case, the set of authors) directly. When you use `yield from` with a set, it effectively "delegates" the yielding operation to the set, so each unique author in the set is yielded one at a time.

### Why Use This Code?
- **Efficiency**: Using a set comprehension ensures that you only get unique authors, eliminating any duplicates from the collection of books.
- **Generators**: This is a generator way of producing values. By using `yield from`, the function can be paused and resumed, providing a way to process or consume the authors one at a time, which can be more memory efficient, especially if the collection of books is large.

### Summary
The code creates a generator that produces each unique author from the list of books, skipping any books that do not have an author specified. This is useful in scenarios where you want to work with a unique list of authors efficiently.

In [39]:
# Get Llama 3.2 to answer (separate client — do not reuse the name `client_openai`)
client_ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [40]:
def stream_answer_with_ollama():
    stream = client_ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages= [
            {"role": "system", "content": "You are an assisstant that will give explanation of the question asked"},
            {"role": "user", "content": question}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id = display_handle.display_id)

In [41]:
stream_answer(question)

This code snippet uses Python's `yield from` syntax in conjunction with a set comprehension. Let's break down what each part does:

1. **Set Comprehension**: 
   - `{book.get("author") for book in books if book.get("author")}` is a set comprehension that creates a set of authors from a collection named `books`.
   - Here, `books` is expected to be an iterable (like a list or a set) of dictionaries, where each dictionary represents a book with its attributes (e.g., title, author, etc.).
   - The expression `book.get("author")` is used to fetch the value associated with the key `"author"` in each `book` dictionary.
   - The `if book.get("author")` condition ensures that only books with a defined author (i.e., where `book.get("author")` does not return `None` or an empty string) are included in the set.

2. **Set Behavior**:
   - Since this is a set comprehension, the resulting set will contain unique author names (i.e., if two books have the same author, that author will only appear once in the set).

3. **Yield from**: 
   - The `yield from` statement is used in generator functions to yield all the values from an iterable, in this case, the set that was created.
   - This means that instead of returning the entire set as a single value, the generator will produce each author name one at a time as it is being iterated over.

### Purpose
Overall, this piece of code is part of a generator that yields the unique authors of the books from the `books` collection. It allows for efficient retrieval and processing of unique authors, particularly useful when you want to process or list authors without creating duplicates.

### Example
Consider you have a list of books like this:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},
    {"title": "Book 4", "author": None},
    {"title": "Book 5", "author": ""},
]
```

When you run the code:

```python
# Example usage of the generator function
def get_unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# You can collect unique authors like so
for author in get_unique_authors(books):
    print(author)
```

The output would be:

```
Author A
Author B
```

This shows that "Author A" and "Author B" are the unique authors, while the entries with no author or an empty string are excluded.